# 🏭 Notebook 04 — Evaluation & Comparison
## PatchCore vs Convolutional Autoencoder | MVTec AD Benchmark

**Purpose:** Rigorous evaluation with strict no-leakage protocol  
**Metrics:** AUROC · Average Precision · F1 · FPR · Pixel-AUROC  
**Experiments:** 2 methods × 3 categories = 6 experiments  

**Data flow:**
```
val_indices.json  → threshold selection ONLY
test holdout      → metric evaluation ONLY
No overlap. No leakage.
```
---

### PRE-STEP — Install & Imports

In [1]:
# ─────────────────────────────────────────────────────────────────
# PRE-STEP — Install missing packages
# ─────────────────────────────────────────────────────────────────
import importlib, subprocess, sys

def install_if_missing(module_name, pip_name):
    if importlib.util.find_spec(module_name) is None:
        print(f'📦 Installing {pip_name}...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install',
                            '--quiet', pip_name],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f'❌ Failed: {pip_name}\n{r.stderr}')
        print(f'  ✅ {pip_name} installed.')
    else:
        print(f'  ⏭️  {pip_name} already available.')

install_if_missing('psutil', 'psutil')
print('\n✅ Dependencies ready.')

  ⏭️  psutil already available.

✅ Dependencies ready.


In [2]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 — Imports
# ─────────────────────────────────────────────────────────────────
import os, json, gc, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    roc_curve,
)
import cv2
import psutil

warnings.filterwarnings('ignore')
print('✅ Imports complete.')

✅ Imports complete.


---
### STEP 1 — Load Config & All Scores

In [3]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — Load config + all scores from Notebook 02 & 03
# ─────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_NAME = 'industrial_anomaly_project'
DRIVE_ROOT   = f'/content/drive/MyDrive/{PROJECT_NAME}'
CONFIG_PATH  = f'{DRIVE_ROOT}/configs/config.json'

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f'❌ config.json not found. Run Notebook 00 first.')

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

RESULTS_DRIVE  = cfg['RESULTS_DRIVE']
CONFIG_DRIVE   = cfg['CONFIG_DRIVE']
LOCAL_DATA     = cfg['LOCAL_DATA']
DATASET_DRIVE  = cfg['DATASET_DRIVE']
CATEGORIES     = cfg['CATEGORIES']
SEED           = cfg['SEED']
IMAGE_SIZE     = cfg['IMAGE_SIZE']
VAL_SPLIT      = cfg['VAL_SPLIT']

os.makedirs(RESULTS_DRIVE, exist_ok=True)

# ── Load val_indices (LOCKED from Notebook 01) ───────────────────
VAL_INDICES_PATH = f'{CONFIG_DRIVE}/val_indices.json'
if not os.path.exists(VAL_INDICES_PATH):
    raise FileNotFoundError(f'❌ val_indices.json not found: {VAL_INDICES_PATH}')
with open(VAL_INDICES_PATH) as f:
    VAL_INDICES = json.load(f)

# ── Load all scores ───────────────────────────────────────────────
# Structure: DATA[method][category] = {scores, maps, labels, paths, defect_types}
DATA = {'patchcore': {}, 'autoencoder': {}}

print('📂 Loading scores and maps...\n')

for cat in CATEGORIES:
    # PatchCore
    pc_scores = np.load(f'{RESULTS_DRIVE}/patchcore_{cat}_scores.npy')
    pc_maps   = np.load(f'{RESULTS_DRIVE}/patchcore_{cat}_maps.npy')   # (N, 32, 32)
    pc_labels = np.load(f'{RESULTS_DRIVE}/patchcore_{cat}_labels.npy')
    with open(f'{RESULTS_DRIVE}/patchcore_{cat}_meta.json') as f:
        pc_meta = json.load(f)
    DATA['patchcore'][cat] = {
        'scores': pc_scores, 'maps': pc_maps,
        'labels': pc_labels, 'paths': pc_meta['paths'],
        'defect_types': pc_meta['defect_types'],
    }

    # Autoencoder (use mean score as primary)
    ae_scores = np.load(f'{RESULTS_DRIVE}/ae_{cat}_scores.npy')
    ae_maps   = np.load(f'{RESULTS_DRIVE}/ae_{cat}_maps.npy')          # (N, 256, 256)
    ae_labels = np.load(f'{RESULTS_DRIVE}/ae_{cat}_labels.npy')
    with open(f'{RESULTS_DRIVE}/ae_{cat}_meta.json') as f:
        ae_meta = json.load(f)
    DATA['autoencoder'][cat] = {
        'scores': ae_scores, 'maps': ae_maps,
        'labels': ae_labels, 'paths': ae_meta['paths'],
        'defect_types': ae_meta['defect_types'],
    }

    print(f'  ✅ {cat}')
    print(f'     PatchCore : scores{pc_scores.shape} maps{pc_maps.shape} '
          f'normal={int((pc_labels==0).sum())} defective={int((pc_labels==1).sum())}')
    print(f'     AE        : scores{ae_scores.shape} maps{ae_maps.shape} '
          f'normal={int((ae_labels==0).sum())} defective={int((ae_labels==1).sum())}')
    print()

print('✅ All scores loaded.')

Mounted at /content/drive
📂 Loading scores and maps...

  ✅ leather
     PatchCore : scores(104,) maps(104, 32, 32) normal=22 defective=82
     AE        : scores(104,) maps(104, 256, 256) normal=22 defective=82

  ✅ tile
     PatchCore : scores(97,) maps(97, 32, 32) normal=23 defective=74
     AE        : scores(97,) maps(97, 256, 256) normal=23 defective=74

  ✅ metal_nut
     PatchCore : scores(100,) maps(100, 32, 32) normal=17 defective=83
     AE        : scores(100,) maps(100, 256, 256) normal=17 defective=83

✅ All scores loaded.


In [7]:
# ─────────────────────────────────────────────────────────────────
# STEP 1b — FIXED: Build val scores directly from val image paths
#
# Root cause: result meta only contains HOLDOUT images (val was
# excluded during scoring in NB02/NB03). So val images are NOT
# in result paths — we cannot build a mask from them.
#
# Fix: Treat the entire result array as test holdout (it already is).
# For threshold selection, we need val scores separately.
# Solution: re-score val images using saved model artifacts.
#
# SIMPLER APPROACH for threshold:
#   Use the holdout scores directly split by label to simulate
#   a threshold search, OR re-extract val scores from scratch.
#
# We choose the cleanest academic approach:
#   Load all test image paths + scores, then re-split using
#   val_indices to get val subset.
#
# But since val was excluded from scoring, we must re-score them.
# For PatchCore: load memory bank + FAISS, score val images.
# For AE: load checkpoint, score val images.
# ─────────────────────────────────────────────────────────────────

# ── Verify current state: result arrays are pure holdout ─────────
print('📋 Confirming result arrays are pure holdout (no val images):\n')
for method in ['patchcore', 'autoencoder']:
    for cat in CATEGORIES:
        result_paths = DATA[method][cat]['paths']
        val_paths    = set(
            VAL_INDICES[cat]['normal'] +
            VAL_INDICES[cat]['defective']
        )
        overlap = set(result_paths) & val_paths
        n_val_expected = VAL_SPLIT[cat]['normal'] + VAL_SPLIT[cat]['defective']
        print(f'  [{method}][{cat}]: holdout={len(result_paths)} '
              f'| val_overlap={len(overlap)} '
              f'| val_expected={n_val_expected} '
              f'{"✅ clean" if len(overlap)==0 else "⚠️ has overlap"}')

print()
print('✅ Confirmed: result arrays contain ONLY holdout images.')
print('   Val images were correctly excluded during NB02/NB03 scoring.')
print()
print('📌 Implication for threshold selection (STEP 3):')
print('   Val images must be re-scored using saved model artifacts.')
print('   This will be handled in STEP 3 using memory banks + AE checkpoints.')

📋 Confirming result arrays are pure holdout (no val images):

  [patchcore][leather]: holdout=104 | val_overlap=0 | val_expected=20 ✅ clean
  [patchcore][tile]: holdout=97 | val_overlap=0 | val_expected=20 ✅ clean
  [patchcore][metal_nut]: holdout=100 | val_overlap=0 | val_expected=15 ✅ clean
  [autoencoder][leather]: holdout=104 | val_overlap=0 | val_expected=20 ✅ clean
  [autoencoder][tile]: holdout=97 | val_overlap=0 | val_expected=20 ✅ clean
  [autoencoder][metal_nut]: holdout=100 | val_overlap=0 | val_expected=15 ✅ clean

✅ Confirmed: result arrays contain ONLY holdout images.
   Val images were correctly excluded during NB02/NB03 scoring.

📌 Implication for threshold selection (STEP 3):
   Val images must be re-scored using saved model artifacts.
   This will be handled in STEP 3 using memory banks + AE checkpoints.


---
### STEP 2 — Score Normalization

In [9]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — Score Normalization (FIXED)
#
# Result arrays are PURE HOLDOUT (val already excluded in NB02/03).
# So we fit min-max on the FULL result array — it IS the holdout.
# No val_mask needed here.
# ─────────────────────────────────────────────────────────────────

def normalize_scores(scores: np.ndarray) -> tuple:
    """
    Min-max normalize scores.
    Fits and applies on the same array (which is pure holdout).

    Returns:
        scores_norm : (N,) normalized to [0,1]
        norm_params : dict with 'min', 'max'
    """
    s_min = float(scores.min())
    s_max = float(scores.max())
    denom = (s_max - s_min) if (s_max - s_min) > 1e-8 else 1e-8
    scores_norm = np.clip((scores - s_min) / denom, 0.0, 1.0)
    return scores_norm, {'min': s_min, 'max': s_max}


SCORES_NORM = {}
NORM_PARAMS = {}

print('⚖️  Normalizing scores (result arrays are pure holdout)...\n')

for method in ['patchcore', 'autoencoder']:
    SCORES_NORM[method] = {}
    NORM_PARAMS[method] = {}
    for cat in CATEGORIES:
        scores = DATA[method][cat]['scores']
        scores_norm, params = normalize_scores(scores)
        SCORES_NORM[method][cat] = scores_norm
        NORM_PARAMS[method][cat] = params

        print(f'  [{method}][{cat}]')
        print(f'     raw  : [{scores.min():.6f}, {scores.max():.6f}]')
        print(f'     norm : [{scores_norm.min():.4f}, {scores_norm.max():.4f}]')
        print()

# Save normalization params
THRESHOLDS_PATH = f'{CONFIG_DRIVE}/thresholds.json'
thresholds_data = {'norm_params': NORM_PARAMS}
with open(THRESHOLDS_PATH, 'w') as f:
    json.dump(thresholds_data, f, indent=4)

print('✅ Normalization complete. Params saved.')

⚖️  Normalizing scores (result arrays are pure holdout)...

  [patchcore][leather]
     raw  : [8.232121, 50.115414]
     norm : [0.0000, 1.0000]

  [patchcore][tile]
     raw  : [14.688721, 49.628704]
     norm : [0.0000, 1.0000]

  [patchcore][metal_nut]
     raw  : [13.113266, 39.361958]
     norm : [0.0000, 1.0000]

  [autoencoder][leather]
     raw  : [0.000651, 0.001957]
     norm : [0.0000, 1.0000]

  [autoencoder][tile]
     raw  : [0.000766, 0.003059]
     norm : [0.0000, 1.0000]

  [autoencoder][metal_nut]
     raw  : [0.000439, 0.001111]
     norm : [0.0000, 1.0000]

✅ Normalization complete. Params saved.


In [13]:
# ─────────────────────────────────────────────────────────────────
# STEP 2b — Restore local dataset (run after every session reset)
# ─────────────────────────────────────────────────────────────────

import shutil, time

print('🔍 Checking local dataset...\n')

for cat in CATEGORIES:
    src  = os.path.join(DATASET_DRIVE, cat)
    dest = os.path.join(LOCAL_DATA, cat)
    dest_ok = (
        os.path.exists(dest) and
        os.path.exists(os.path.join(dest, 'train', 'good')) and
        len(os.listdir(os.path.join(dest, 'train', 'good'))) > 0
    )
    if dest_ok:
        n = len(os.listdir(os.path.join(dest, 'train', 'good')))
        print(f'  ⏭️  {cat:<12}: already at LOCAL_DATA ({n} images) — skipped.')
        continue
    if not os.path.exists(src):
        raise FileNotFoundError(f'❌ Not on Drive: {src}. Re-run Notebook 01.')
    t0 = time.time()
    print(f'  📋 {cat:<12}: copying from Drive...', end='', flush=True)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    n = len(os.listdir(os.path.join(dest, 'train', 'good')))
    print(f' done in {time.time()-t0:.1f}s ({n} train images)')

print('\n✅ Dataset ready at LOCAL_DATA.')

🔍 Checking local dataset...

  📋 leather     : copying from Drive... done in 73.0s (245 train images)
  📋 tile        : copying from Drive... done in 61.3s (230 train images)
  📋 metal_nut   : copying from Drive... done in 55.5s (220 train images)

✅ Dataset ready at LOCAL_DATA.


---
### STEP 3 — Threshold Selection (Validation Only)

In [14]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — Re-score val images + Threshold selection
#
# Val images were excluded from NB02/NB03 scoring (correct design).
# Re-score them here using saved artifacts for threshold selection.
# ─────────────────────────────────────────────────────────────────

import importlib, subprocess, sys, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as T
import numpy as np
import faiss
from PIL import Image

# ── Install faiss if missing ──────────────────────────────────────
def install_if_missing(module_name, pip_name):
    if importlib.util.find_spec(module_name) is None:
        print(f'📦 Installing {pip_name}...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', pip_name],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            raise RuntimeError(f'❌ Failed: {pip_name}\n{r.stderr}')
        print(f'  ✅ {pip_name} installed.')
    else:
        print(f'  ⏭️  {pip_name} already available.')

install_if_missing('faiss', 'faiss-cpu')

# ── Unpack config (safe after session reset) ──────────────────────
MODEL_DRIVE   = cfg['MODEL_DRIVE']
LOCAL_MODELS  = cfg['LOCAL_MODELS']
DATASET_DRIVE = cfg['DATASET_DRIVE']
LOCAL_DATA    = cfg['LOCAL_DATA']
IMAGE_SIZE    = cfg['IMAGE_SIZE']
IMAGENET_MEAN = cfg['IMAGENET_MEAN']
IMAGENET_STD  = cfg['IMAGENET_STD']
KNN_K         = cfg['PATCHCORE_KNN_K']
PATCH_DIM     = 1536

os.makedirs(LOCAL_MODELS, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {DEVICE}')

# ── Image transform ───────────────────────────────────────────────
TRANSFORM_VAL = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE),
             interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

_MEAN = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
_STD  = torch.tensor(IMAGENET_STD ).view(1, 3, 1, 1)

def denormalize(x: torch.Tensor) -> torch.Tensor:
    return torch.clamp(
        x * _STD.to(x.device) + _MEAN.to(x.device), 0.0, 1.0
    )

def load_image_tensor(path: str) -> torch.Tensor:
    img = Image.open(path).convert('RGB')
    return TRANSFORM_VAL(img).unsqueeze(0)  # (1, 3, H, W)


# ── PatchCore feature extractor ───────────────────────────────────
class PatchCoreExtractorEval:
    """WideResNet-50 feature extractor for val image scoring."""

    def __init__(self, device):
        self.device   = device
        self.features = {}
        backbone = tv_models.wide_resnet50_2(
            weights=tv_models.Wide_ResNet50_2_Weights.IMAGENET1K_V1
        )
        for p in backbone.parameters():
            p.requires_grad = False
        backbone.eval().to(device)
        backbone.layer2.register_forward_hook(
            lambda m, i, o: self.features.__setitem__('layer2', o)
        )
        backbone.layer3.register_forward_hook(
            lambda m, i, o: self.features.__setitem__('layer3', o)
        )
        self.model = backbone

    @torch.no_grad()
    def extract(self, x: torch.Tensor) -> torch.Tensor:
        self.features.clear()
        _ = self.model(x.to(self.device))
        f2    = self.features['layer2']
        f3    = self.features['layer3']
        f3_up = F.interpolate(
            f3, size=f2.shape[-2:], mode='bilinear', align_corners=False
        )
        fused    = torch.cat([f2, f3_up], dim=1)  # (B, 1536, H, W)
        B, C, H, W = fused.shape
        return fused.permute(0, 2, 3, 1).reshape(B, H * W, C).cpu()


def score_patchcore_single(img_tensor, extractor, faiss_index, k=9):
    """Score one image: max kNN distance across all patches."""
    patches  = extractor.extract(img_tensor).numpy()[0]  # (N_patches, 1536)
    dists, _ = faiss_index.search(patches.astype(np.float32), k)
    return float(dists[:, 0].max())


# ── Convolutional Autoencoder ─────────────────────────────────────
class ConvAE(nn.Module):
    """Must match architecture used in Notebook 03."""

    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            self._enc(  3,  32), self._enc( 32,  64),
            self._enc( 64, 128), self._enc(128, 256),
        )
        self.decoder = nn.Sequential(
            self._dec(256, 128), self._dec(128,  64),
            self._dec( 64,  32),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Sigmoid(),
        )

    @staticmethod
    def _enc(i, o):
        return nn.Sequential(
            nn.Conv2d(i, o, 3, 2, 1, bias=False),
            nn.BatchNorm2d(o),
            nn.LeakyReLU(0.2, inplace=True),
        )

    @staticmethod
    def _dec(i, o):
        return nn.Sequential(
            nn.ConvTranspose2d(i, o, 4, 2, 1, bias=False),
            nn.BatchNorm2d(o),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


@torch.no_grad()
def score_ae_single(img_tensor, model):
    """Score one image: mean pixel reconstruction error."""
    img_gpu = img_tensor.to(DEVICE)
    recon   = model(img_gpu)
    target  = denormalize(img_gpu)
    err_map = ((recon - target) ** 2).mean(dim=1)  # (1, H, W)
    return float(err_map.mean().item())


# ── Main: re-score all val images ────────────────────────────────
print('\n🔄 Re-scoring validation images...\n')

VAL_SCORES   = {'patchcore': {}, 'autoencoder': {}}
pc_extractor = None  # lazy init — loaded once, reused across categories

for cat in CATEGORIES:
    val_paths  = VAL_INDICES[cat]['normal'] + VAL_INDICES[cat]['defective']
    val_labels = (
        [0] * len(VAL_INDICES[cat]['normal']) +
        [1] * len(VAL_INDICES[cat]['defective'])
    )
    print(f'  [{cat}] {len(val_paths)} val images '
          f'(normal={len(VAL_INDICES[cat]["normal"])} '
          f'defective={len(VAL_INDICES[cat]["defective"])})')

    # ── PatchCore scoring ─────────────────────────────────────────
    bank  = np.load(
        os.path.join(MODEL_DRIVE, f'patchcore_{cat}.npy')
    ).astype(np.float32)
    index = faiss.IndexFlatL2(bank.shape[1])
    index.add(bank)

    if pc_extractor is None:
        print('  Loading WideResNet-50 (once for all categories)...')
        pc_extractor = PatchCoreExtractorEval(DEVICE)

    pc_scores = [
        score_patchcore_single(load_image_tensor(p), pc_extractor, index, KNN_K)
        for p in val_paths
    ]
    VAL_SCORES['patchcore'][cat] = {
        'scores': np.array(pc_scores, dtype=np.float32),
        'labels': np.array(val_labels, dtype=np.int32),
    }
    del bank, index
    gc.collect()
    print(f'     PatchCore   ✅  scored {len(pc_scores)} images')

    # ── Autoencoder scoring ───────────────────────────────────────
    ae_model = ConvAE().to(DEVICE)
    ckpt     = torch.load(
        os.path.join(MODEL_DRIVE, f'autoencoder_{cat}.pt'),
        map_location=DEVICE
    )
    ae_model.load_state_dict(ckpt['model_state'])
    ae_model.eval()

    ae_scores = [
        score_ae_single(load_image_tensor(p), ae_model)
        for p in val_paths
    ]
    VAL_SCORES['autoencoder'][cat] = {
        'scores': np.array(ae_scores, dtype=np.float32),
        'labels': np.array(val_labels, dtype=np.int32),
    }
    del ae_model, ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'     AutoEncoder ✅  scored {len(ae_scores)} images\n')

print('✅ All val images re-scored.')

  ⏭️  faiss-cpu already available.

Device: cpu

🔄 Re-scoring validation images...

  [leather] 20 val images (normal=10 defective=10)
  Loading WideResNet-50 (once for all categories)...
     PatchCore   ✅  scored 20 images
     AutoEncoder ✅  scored 20 images

  [tile] 20 val images (normal=10 defective=10)
     PatchCore   ✅  scored 20 images
     AutoEncoder ✅  scored 20 images

  [metal_nut] 15 val images (normal=5 defective=10)
     PatchCore   ✅  scored 15 images
     AutoEncoder ✅  scored 15 images

✅ All val images re-scored.


In [18]:
# ─────────────────────────────────────────────────────────────────
# STEP 3b — Find optimal threshold from val scores
# ─────────────────────────────────────────────────────────────────

THRESHOLDS = {}

print('🎯 Finding optimal thresholds on validation set...\n')

for method in ['patchcore', 'autoencoder']:
    THRESHOLDS[method] = {}
    for cat in CATEGORIES:
        s_min  = NORM_PARAMS[method][cat]['min']
        s_max  = NORM_PARAMS[method][cat]['max']
        denom  = (s_max - s_min) if (s_max - s_min) > 1e-8 else 1e-8

        val_raw    = VAL_SCORES[method][cat]['scores']
        val_norm   = np.clip((val_raw - s_min) / denom, 0.0, 1.0)
        val_labels = VAL_SCORES[method][cat]['labels']

        thresholds_arr = np.linspace(0.0, 1.0, 1000)
        f1_scores      = np.zeros(len(thresholds_arr), dtype=np.float32)

        for i, t in enumerate(thresholds_arr):
            preds = (val_norm >= t).astype(int)
            tp    = int(((preds == 1) & (val_labels == 1)).sum())
            fp    = int(((preds == 1) & (val_labels == 0)).sum())
            fn    = int(((preds == 0) & (val_labels == 1)).sum())
            d     = 2 * tp + fp + fn
            f1_scores[i] = (2 * tp / d) if d > 0 else 0.0

        best_idx  = int(np.argmax(f1_scores))
        threshold = float(thresholds_arr[best_idx])
        val_f1    = float(f1_scores[best_idx])
        THRESHOLDS[method][cat] = threshold

        print(f'  [{method}][{cat}]')
        print(f'     val samples : {len(val_labels)} '
              f'(normal={int((val_labels==0).sum())} '
              f'defective={int((val_labels==1).sum())})')
        print(f'     norm range  : [{val_norm.min():.4f}, {val_norm.max():.4f}]')
        print(f'     threshold   : {threshold:.4f}  |  val F1: {val_f1:.4f}')
        print()

thresholds_data['optimal_thresholds'] = THRESHOLDS
with open(THRESHOLDS_PATH, 'w') as f:
    json.dump(thresholds_data, f, indent=4)

print(f'✅ Thresholds saved to: {THRESHOLDS_PATH}')

🎯 Finding optimal thresholds on validation set...

  [patchcore][leather]
     val samples : 20 (normal=10 defective=10)
     norm range  : [0.0096, 1.0000]
     threshold   : 0.0591  |  val F1: 1.0000

  [patchcore][tile]
     val samples : 20 (normal=10 defective=10)
     norm range  : [0.0000, 0.6135]
     threshold   : 0.0761  |  val F1: 1.0000

  [patchcore][metal_nut]
     val samples : 15 (normal=5 defective=10)
     norm range  : [0.0395, 1.0000]
     threshold   : 0.1151  |  val F1: 1.0000

  [autoencoder][leather]
     val samples : 20 (normal=10 defective=10)
     norm range  : [0.0000, 0.5906]
     threshold   : 0.3223  |  val F1: 0.7000

  [autoencoder][tile]
     val samples : 20 (normal=10 defective=10)
     norm range  : [0.0153, 0.3214]
     threshold   : 0.0641  |  val F1: 0.7273

  [autoencoder][metal_nut]
     val samples : 15 (normal=5 defective=10)
     norm range  : [0.1259, 0.9269]
     threshold   : 0.0000  |  val F1: 0.8000

✅ Thresholds saved to: /content/dri

---
### STEP 4 — Evaluation on Test Holdout

In [19]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — compute_metrics() (FIXED — no val_mask)
#
# Result arrays are pure holdout. Use directly.
# ─────────────────────────────────────────────────────────────────

IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}


def load_gt_masks(paths, labels, defect_types, category,
                  target_size=256):
    masks = []
    gt_base = None
    for data_root in [LOCAL_DATA, DATASET_DRIVE]:
        candidate = Path(data_root) / category / 'ground_truth'
        if candidate.exists():
            gt_base = candidate
            break

    for img_path, label, defect_type in zip(paths, labels, defect_types):
        if label == 0:
            masks.append(np.zeros((target_size, target_size),
                                  dtype=np.float32))
            continue

        img_p     = Path(img_path)
        mask_name = f'{img_p.stem}_mask{img_p.suffix}'
        mask_path = gt_base / defect_type / mask_name \
                    if gt_base else None

        if mask_path is None or not mask_path.exists():
            fb = (Path(DATASET_DRIVE) / category /
                  'ground_truth' / defect_type / mask_name)
            mask_path = fb if fb.exists() else None

        if mask_path is None:
            masks.append(np.zeros((target_size, target_size),
                                  dtype=np.float32))
            continue

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            masks.append(np.zeros((target_size, target_size),
                                  dtype=np.float32))
            continue

        mask = cv2.resize(mask, (target_size, target_size),
                          interpolation=cv2.INTER_NEAREST)
        masks.append((mask > 127).astype(np.float32))

    return np.stack(masks, axis=0)


def upsample_maps(maps: np.ndarray, target_size: int = 256):
    if maps.shape[1] == target_size and maps.shape[2] == target_size:
        return maps
    return np.stack([
        cv2.resize(m, (target_size, target_size),
                   interpolation=cv2.INTER_LINEAR)
        for m in maps
    ], axis=0)


def compute_metrics(method, category, scores_norm, labels,
                    maps, paths, defect_types, threshold):
    """
    Compute all metrics on test holdout.
    Result arrays are pure holdout — use directly (no val_mask).
    """
    # Pure holdout — use as-is
    h_scores  = scores_norm
    h_labels  = labels
    h_maps    = maps
    h_paths   = paths
    h_dtypes  = defect_types

    assert h_labels.sum() > 0, \
        f'❌ No defective samples in [{method}][{category}]'
    assert (h_labels == 0).sum() > 0, \
        f'❌ No normal samples in [{method}][{category}]'

    # ── Image-level metrics ───────────────────────────────────────
    auroc = float(roc_auc_score(h_labels, h_scores))
    ap    = float(average_precision_score(h_labels, h_scores))

    preds    = (h_scores >= threshold).astype(int)
    f1       = float(f1_score(h_labels, preds, zero_division=0))
    n_normal = int((h_labels == 0).sum())
    fp       = int(((preds == 1) & (h_labels == 0)).sum())
    fpr      = float(fp / n_normal) if n_normal > 0 else 0.0

    # ── Pixel-AUROC ───────────────────────────────────────────────
    maps_256 = upsample_maps(h_maps, target_size=IMAGE_SIZE)
    gt_masks = load_gt_masks(
        h_paths, h_labels, h_dtypes, category,
        target_size=IMAGE_SIZE
    )

    flat_gt   = gt_masks.flatten()
    flat_maps = maps_256.flatten()
    m_min, m_max = flat_maps.min(), flat_maps.max()
    if m_max - m_min > 1e-8:
        flat_maps_norm = (flat_maps - m_min) / (m_max - m_min)
    else:
        flat_maps_norm = flat_maps

    if flat_gt.sum() > 0:
        pixel_auroc = float(roc_auc_score(flat_gt, flat_maps_norm))
    else:
        pixel_auroc = float('nan')
        print(f'  ⚠️  [{method}][{category}]: no defect pixels in GT masks')

    return {
        'method':       method,
        'category':     category,
        'auroc':        round(auroc, 4),
        'ap':           round(ap, 4),
        'f1':           round(f1, 4),
        'fpr':          round(fpr, 4),
        'pixel_auroc':  round(pixel_auroc, 4)
                        if not np.isnan(pixel_auroc) else None,
        'threshold':    round(threshold, 4),
        'n_holdout':    int(len(h_labels)),
        'n_normal':     int((h_labels == 0).sum()),
        'n_defective':  int((h_labels == 1).sum()),
    }


print('✅ compute_metrics() defined.')

✅ compute_metrics() defined.


---
### STEP 5 — Per-Defect-Type Analysis

In [20]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — per_defect_analysis() (FIXED — no val_mask)
# ─────────────────────────────────────────────────────────────────

def per_defect_analysis(scores_norm, labels, defect_types, threshold):
    """F1 per defect type. Uses full result array (pure holdout)."""
    preds = (scores_norm >= threshold).astype(int)

    defect_results = defaultdict(
        lambda: {'tp': 0, 'fp': 0, 'fn': 0, 'n': 0}
    )

    for pred, label, dtype in zip(preds, labels, defect_types):
        if dtype == 'good':
            continue
        defect_results[dtype]['n'] += 1
        if   label == 1 and pred == 1: defect_results[dtype]['tp'] += 1
        elif label == 1 and pred == 0: defect_results[dtype]['fn'] += 1
        elif label == 0 and pred == 1: defect_results[dtype]['fp'] += 1

    output = {}
    for dtype, c in sorted(defect_results.items()):
        tp, fp, fn = c['tp'], c['fp'], c['fn']
        d  = 2 * tp + fp + fn
        f1 = round((2 * tp / d) if d > 0 else 0.0, 4)
        output[dtype] = {
            'f1': f1, 'n_samples': c['n'],
            'tp': tp, 'fp': fp, 'fn': fn,
        }
    return output


print('✅ per_defect_analysis() defined.')

✅ per_defect_analysis() defined.


---
### STEP 7 — Main Loop: All Experiments

In [21]:
# ─────────────────────────────────────────────────────────────────
# STEP 7 — Main evaluation loop (FIXED — no val_mask)
# ─────────────────────────────────────────────────────────────────

all_results        = []
all_defect_results = {}
METHODS            = ['patchcore', 'autoencoder']

for method in METHODS:
    print()
    print('═' * 60)
    print(f'  Evaluating: {method.upper()}')
    print('═' * 60)

    for cat in CATEGORIES:
        print(f'\n  [{method}][{cat}]')

        scores_norm  = SCORES_NORM[method][cat]
        labels       = DATA[method][cat]['labels']
        maps         = DATA[method][cat]['maps']
        paths        = DATA[method][cat]['paths']
        defect_types = DATA[method][cat]['defect_types']
        threshold    = THRESHOLDS[method][cat]

        # ── Metrics ───────────────────────────────────────────────
        metrics = compute_metrics(
            method, cat,
            scores_norm, labels, maps,
            paths, defect_types, threshold
        )
        all_results.append(metrics)

        print(f'     AUROC       : {metrics["auroc"]:.4f}')
        print(f'     AP          : {metrics["ap"]:.4f}')
        print(f'     F1          : {metrics["f1"]:.4f}'
              f'  (threshold={threshold:.4f})')
        print(f'     FPR         : {metrics["fpr"]:.4f}')
        print(f'     Pixel-AUROC : {metrics["pixel_auroc"]}')
        print(f'     Holdout     : {metrics["n_holdout"]} '
              f'(normal={metrics["n_normal"]} '
              f'defective={metrics["n_defective"]})')

        # ── Per-defect breakdown ──────────────────────────────────
        defect_breakdown = per_defect_analysis(
            scores_norm, labels, defect_types, threshold
        )
        key = f'{method}_{cat}'
        all_defect_results[key] = defect_breakdown

        print(f'     Per-defect F1:')
        for dtype, dr in defect_breakdown.items():
            print(f'       {dtype:<18}: '
                  f'F1={dr["f1"]:.4f}  n={dr["n_samples"]}')

print()
print('═' * 60)
print(f'  Evaluation complete: {len(all_results)} experiments')
print('═' * 60)


════════════════════════════════════════════════════════════
  Evaluating: PATCHCORE
════════════════════════════════════════════════════════════

  [patchcore][leather]
     AUROC       : 1.0000
     AP          : 1.0000
     F1          : 0.9939  (threshold=0.0591)
     FPR         : 0.0455
     Pixel-AUROC : 0.9945
     Holdout     : 104 (normal=22 defective=82)
     Per-defect F1:
       color             : F1=1.0000  n=16
       cut               : F1=1.0000  n=17
       fold              : F1=1.0000  n=16
       glue              : F1=1.0000  n=17
       poke              : F1=1.0000  n=16

  [patchcore][tile]
     AUROC       : 0.9342
     AP          : 0.9779
     F1          : 0.9281  (threshold=0.0761)
     FPR         : 0.3478
     Pixel-AUROC : 0.9458
     Holdout     : 97 (normal=23 defective=74)
     Per-defect F1:
       crack             : F1=1.0000  n=15
       glue_strip        : F1=1.0000  n=15
       gray_stroke       : F1=0.9231  n=14
       oil               : F1

---
### STEP 6 — Save experiment_log.csv

In [22]:
# ─────────────────────────────────────────────────────────────────
# STEP 6 — Save experiment_log.csv
# ─────────────────────────────────────────────────────────────────

CSV_PATH = f'{RESULTS_DRIVE}/experiment_log.csv'

# Build DataFrame with clean column order
df = pd.DataFrame(all_results)[[
    'method', 'category', 'auroc', 'ap', 'f1', 'fpr',
    'pixel_auroc', 'threshold', 'n_holdout', 'n_normal', 'n_defective'
]]

df.to_csv(CSV_PATH, index=False, float_format='%.4f')

print(f'✅ experiment_log.csv saved to: {CSV_PATH}\n')
print(df.to_string(index=False))

# Also save per-defect results
DEFECT_PATH = f'{RESULTS_DRIVE}/per_defect_results.json'
with open(DEFECT_PATH, 'w') as f:
    json.dump(all_defect_results, f, indent=4)
print(f'\n✅ Per-defect breakdown saved to: {DEFECT_PATH}')

✅ experiment_log.csv saved to: /content/drive/MyDrive/industrial_anomaly_project/results/experiment_log.csv

     method  category  auroc     ap     f1    fpr  pixel_auroc  threshold  n_holdout  n_normal  n_defective
  patchcore   leather 1.0000 1.0000 0.9939 0.0455       0.9945     0.0591        104        22           82
  patchcore      tile 0.9342 0.9779 0.9281 0.3478       0.9458     0.0761         97        23           74
  patchcore metal_nut 0.9972 0.9994 0.9765 0.2353       0.9807     0.1151        100        17           83
autoencoder   leather 0.5615 0.8677 0.7083 0.5000       0.8326     0.3223        104        22           82
autoencoder      tile 0.8813 0.9623 0.8974 0.5217       0.6837     0.0641         97        23           74
autoencoder metal_nut 0.2991 0.7313 0.9071 1.0000       0.7282     0.0000        100        17           83

✅ Per-defect breakdown saved to: /content/drive/MyDrive/industrial_anomaly_project/results/per_defect_results.json


---
### STEP 8 — Aggregate Summary

In [23]:
# ─────────────────────────────────────────────────────────────────
# STEP 8 — Aggregate summary per method
# ─────────────────────────────────────────────────────────────────

summary = {}

for method in METHODS:
    method_rows = df[df['method'] == method]
    summary[method] = {
        'mean_auroc':       round(float(method_rows['auroc'].mean()), 4),
        'mean_ap':          round(float(method_rows['ap'].mean()), 4),
        'mean_f1':          round(float(method_rows['f1'].mean()), 4),
        'mean_fpr':         round(float(method_rows['fpr'].mean()), 4),
        'mean_pixel_auroc': round(float(method_rows['pixel_auroc'].dropna().mean()), 4),
        'per_category': {
            row['category']: {
                'auroc':       row['auroc'],
                'ap':          row['ap'],
                'f1':          row['f1'],
                'fpr':         row['fpr'],
                'pixel_auroc': row['pixel_auroc'],
            }
            for _, row in method_rows.iterrows()
        }
    }

SUMMARY_PATH = f'{RESULTS_DRIVE}/summary_results.json'
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=4)

print('✅ Summary results saved.\n')

# ── Pretty-print comparison ───────────────────────────────────────
print('=' * 65)
print('  FINAL RESULTS — PatchCore vs Autoencoder')
print('=' * 65)

metrics_to_show = ['auroc', 'ap', 'f1', 'fpr', 'pixel_auroc']

for cat in CATEGORIES:
    print(f'\n  [{cat.upper()}]')
    print(f'  {"Metric":<16} {"PatchCore":>12} {"AutoEncoder":>12}')
    print(f'  {"-"*40}')
    for metric in metrics_to_show:
        pc_val = summary['patchcore']['per_category'][cat].get(metric)
        ae_val = summary['autoencoder']['per_category'][cat].get(metric)
        pc_str = f'{pc_val:.4f}' if pc_val is not None else 'N/A'
        ae_str = f'{ae_val:.4f}' if ae_val is not None else 'N/A'
        winner = ''
        if pc_val is not None and ae_val is not None:
            if metric == 'fpr':
                winner = ' ← PC' if pc_val < ae_val else (' ← AE' if ae_val < pc_val else '')
            else:
                winner = ' ← PC' if pc_val > ae_val else (' ← AE' if ae_val > pc_val else '')
        print(f'  {metric.upper():<16} {pc_str:>12} {ae_str:>12}{winner}')

print(f'\n  {"MEAN":<16} {"PatchCore":>12} {"AutoEncoder":>12}')
print(f'  {"-"*40}')
for metric in ['mean_auroc', 'mean_ap', 'mean_f1', 'mean_fpr', 'mean_pixel_auroc']:
    pc_val = summary['patchcore'][metric]
    ae_val = summary['autoencoder'][metric]
    label  = metric.replace('mean_', '').upper()
    winner = ''
    if metric == 'mean_fpr':
        winner = ' ← PC' if pc_val < ae_val else (' ← AE' if ae_val < pc_val else '')
    else:
        winner = ' ← PC' if pc_val > ae_val else (' ← AE' if ae_val > pc_val else '')
    print(f'  {label:<16} {pc_val:>12.4f} {ae_val:>12.4f}{winner}')
print('=' * 65)

✅ Summary results saved.

  FINAL RESULTS — PatchCore vs Autoencoder

  [LEATHER]
  Metric              PatchCore  AutoEncoder
  ----------------------------------------
  AUROC                  1.0000       0.5615 ← PC
  AP                     1.0000       0.8677 ← PC
  F1                     0.9939       0.7083 ← PC
  FPR                    0.0455       0.5000 ← PC
  PIXEL_AUROC            0.9945       0.8326 ← PC

  [TILE]
  Metric              PatchCore  AutoEncoder
  ----------------------------------------
  AUROC                  0.9342       0.8813 ← PC
  AP                     0.9779       0.9623 ← PC
  F1                     0.9281       0.8974 ← PC
  FPR                    0.3478       0.5217 ← PC
  PIXEL_AUROC            0.9458       0.6837 ← PC

  [METAL_NUT]
  Metric              PatchCore  AutoEncoder
  ----------------------------------------
  AUROC                  0.9972       0.2991 ← PC
  AP                     0.9994       0.7313 ← PC
  F1                     0.97

---
### STEP 9 — Final Cleanup & Verification

In [24]:
# ─────────────────────────────────────────────────────────────────
# STEP 9 — Cleanup + final asset verification
# ─────────────────────────────────────────────────────────────────

# Release large in-memory arrays
del DATA, SCORES_NORM, VAL_MASKS
gc.collect()

# Verify output files
expected_files = [
    f'{CONFIG_DRIVE}/thresholds.json',
    CSV_PATH,
    DEFECT_PATH,
    SUMMARY_PATH,
]

print('🔍 Verifying output files...\n')
all_ok = True
for fp in expected_files:
    exists  = os.path.exists(fp)
    size_kb = os.path.getsize(fp) / 1024 if exists else 0
    icon    = '✅' if exists else '❌'
    print(f'  {icon} {os.path.basename(fp):<42} {size_kb:>8.2f} KB')
    if not exists:
        all_ok = False

if not all_ok:
    raise SystemExit('❌ Some output files missing.')

print()
print('═' * 60)
print('✅ Notebook 04 COMPLETE — Evaluation Done')
print('═' * 60)
print()
print('📌 Assets saved to Drive:')
print(f'   experiment_log.csv   → {CSV_PATH}')
print(f'   summary_results.json → {SUMMARY_PATH}')
print(f'   per_defect_results   → {DEFECT_PATH}')
print(f'   thresholds.json      → {CONFIG_DRIVE}/thresholds.json')
print()
print('📌 Next step: Run 05_visualization.ipynb')

🔍 Verifying output files...

  ✅ thresholds.json                                1.18 KB
  ✅ experiment_log.csv                             0.50 KB
  ✅ per_defect_results.json                        4.19 KB
  ✅ summary_results.json                           1.58 KB

════════════════════════════════════════════════════════════
✅ Notebook 04 COMPLETE — Evaluation Done
════════════════════════════════════════════════════════════

📌 Assets saved to Drive:
   experiment_log.csv   → /content/drive/MyDrive/industrial_anomaly_project/results/experiment_log.csv
   summary_results.json → /content/drive/MyDrive/industrial_anomaly_project/results/summary_results.json
   per_defect_results   → /content/drive/MyDrive/industrial_anomaly_project/results/per_defect_results.json
   thresholds.json      → /content/drive/MyDrive/industrial_anomaly_project/configs/thresholds.json

📌 Next step: Run 05_visualization.ipynb
